# Laya smoke test

Scores the same HTML corpora as `smoke_test.ipynb`, using a Laya checkpoint instead of
the MarkupLM artifact. `PAGE_MODEL_DIR` in the repository's `.env` chooses which one — the same setting the API
loads, so the notebook always measures the checkpoint the product actually serves. The
notebook adapts to whichever it finds:

| Checkpoint | What it is |
|---|---|
| `artifacts/laya_v3` | the checkpoint the API serves; fine-tuned, but see the warning below |
| `artifacts/laya-phishing-v1` | fine-tuned on the same 800-page PhreshPhish sample as MarkupLM `v1` |
| `artifacts/laya_v1` | the published base checkpoint, never trained on phishing |

Both are ModernBERT-large, 421M parameters, 512-token context, and neither sees the DOM
the way MarkupLM does — Laya reads the markup as plain text.

Reference numbers measured earlier on these same corpora:

| | Handwritten | Mendeley AUC | PhreshPhish AUC |
|---|---|---|---|
| Page length alone | — | 0.745 | 0.739 |
| MarkupLM `v1` | 12 / 14 | 0.814 | 0.869 |
| Laya zero-shot | 11 / 14 | 0.724 | not measured |
| Laya fine-tuned | run this notebook | run this notebook | 0.859 (its own split) |

### What laya_v3 does and does not come with

`laya_v3` is the checkpoint `PAGE_MODEL_DIR` points at, so what this notebook measures is
what the live API scores pages with. Its own `metadata.json` is unusually honest about its
state, and that changes how its output here should be read:

- `"status": "uncalibrated_recovery"` and `"note": "Emergency inference artifact only.
  Calibration, test and reload parity are pending."`
- `"test_metrics": {}` — **no held-out test numbers exist.** The 0.884 accuracy / 0.957 AUC
  in its history are from the *validation* split, which is the split that chose this
  checkpoint, so they are optimistic by construction and are not comparable to the test-split
  numbers quoted for the other models below.
- One epoch finished out of three planned (`best_epoch: 1`, 250 optimizer updates).
- `"calibrated": false` with temperature 1.0 and `"threshold_selected": false`, so 0.5 is an
  assumed cut, not a chosen one. Phisang's risk bands (0.4 / 0.6 / 0.8) sit on top of a raw
  probability.

That is exactly why running this notebook is worth doing: the corpora below are the first
numbers for this checkpoint that were not measured on its own training run.

### Reading the fine-tuned model's reported score carefully

`laya-phishing-v1` reports AUC 0.859 on the test split its own training run held out, where
counting characters alone scores 0.823. That is a gain of 0.036 over a baseline that needs
no model at all. MarkupLM `v1`'s 0.869 was measured on a **different** corpus whose length
baseline sits at 0.739, so 0.859 and 0.869 are not directly comparable however similar they
look. Mendeley below is the corpus where both models can be compared on identical pages.


## Setup

`transformers` decorates ModernBERT with `@torch.compile` at import time, and torch refuses
to compile on Python 3.14, so importing the encoder fails before any model is loaded.
`torch.compile` only changes speed, never results, so it is replaced with a pass-through
here. Remove the patch on Python 3.13 or older.

When the checkpoint carries a `metadata.json`, the question is read from it rather than
retyped below. A fine-tuned head answers the exact question it was trained on; asking a
reworded one silently degrades it, and nothing in the output would show why.

To fetch the base checkpoint instead:

```python
from huggingface_hub import snapshot_download
snapshot_download("convaiinnovations/laya", local_dir="artifacts/laya_v1",
                  allow_patterns=["model.safetensors", "encoder/*", "tokenizer/*",
                                  "rl_agent_config.json", "README.md", "eval/results.json"])
```


In [1]:
import json
import os
import sys
from pathlib import Path

import torch

# Must happen before transformers imports ModernBERT, so keep this cell first.
if sys.version_info >= (3, 14):
    def _passthrough_compile(model=None, **kwargs):
        return (lambda fn: fn) if model is None else model
    torch.compile = _passthrough_compile
    print("torch.compile disabled: unsupported on Python", ".".join(map(str, sys.version_info[:2])))

from bs4 import BeautifulSoup, Comment
from laya import Agent

# One switch for the whole project: PAGE_MODEL_DIR in the repository's .env is what
# the API loads, so this notebook reads the same value rather than keeping its own.
# Override for one run with os.environ["PAGE_MODEL_DIR"] before running this cell.
def repo_root():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "ai" / "markuplm").is_dir() and (folder / "backend").is_dir():
            return folder
    return Path.cwd()


ROOT = repo_root()


def configured_model_dir():
    """PAGE_MODEL_DIR from the environment, else from .env, else the API's default."""
    value = os.environ.get("PAGE_MODEL_DIR")
    source = "environment"
    if not value:
        env_file = ROOT / ".env"
        for line in (env_file.read_text(encoding="utf-8").splitlines()
                     if env_file.is_file() else []):
            line = line.strip()
            if line.startswith("PAGE_MODEL_DIR=") and not line.startswith("#"):
                value = line.split("=", 1)[1].strip().strip('"').strip("'")
                source = f"{env_file.name}"
    if not value:
        value, source = "ai/markuplm/artifacts/laya_v3", "built-in default"
    path = Path(value).expanduser()
    return (path if path.is_absolute() else ROOT / path), source


MODEL_DIR, MODEL_DIR_SOURCE = configured_model_dir()
MODEL_DIR = MODEL_DIR.resolve()
if not (MODEL_DIR / "model.safetensors").is_file():
    raise FileNotFoundError(
        f"No Laya checkpoint at {MODEL_DIR} (from {MODEL_DIR_SOURCE}).\n"
        f"Point PAGE_MODEL_DIR in {ROOT / '.env'} at a checkpoint folder, "
        "or see the download snippet above.")

metadata_path = MODEL_DIR / "metadata.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.is_file() else {}
# train_n alone is not the test: laya_v3 records a training history and signature but no
# page counts, and calling it a base checkpoint would misread every number below it.
FINE_TUNED = bool(metadata.get("train_n") or metadata.get("history")
                  or metadata.get("training_signature"))

agent = Agent(str(MODEL_DIR))
print("Model:", MODEL_DIR.name, f"(from {MODEL_DIR_SOURCE})")
if not FINE_TUNED:
    print("base checkpoint: nothing about phishing was trained into it")
else:
    pages = metadata.get("train_n")
    print(f"fine-tuned on {pages if pages else 'an unrecorded number of'} pages, "
          f"best epoch {metadata.get('best_epoch')}")
    test_scores = metadata.get("test_metrics") or {}
    if test_scores:
        print(f"  its own test split: AUC {test_scores.get('auc')}, accuracy {test_scores.get('accuracy')}")
        print(f"  length-only baseline on that split: {metadata.get('baselines', {}).get('length_only_auc_test')}")
    else:
        # No held-out split was kept. Report the validation numbers as validation
        # numbers rather than letting them read as a test result.
        print("  no held-out test split: metadata['test_metrics'] is empty")
        last = (metadata.get("history") or [{}])[-1].get("val") or {}
        if last:
            print(f"  validation split only (selected this checkpoint, so optimistic): "
                  f"AUC {last.get('auc')}, accuracy {last.get('accuracy')}, n={last.get('n')}")

inference = metadata.get("inference") or {}
if metadata.get("status") or not inference.get("calibrated", True):
    print(f"  status: {metadata.get('status', 'unknown')}; "
          f"calibrated={inference.get('calibrated')}, "
          f"threshold_selected={inference.get('threshold_selected')}")
if metadata.get("note"):
    print(f"  note: {metadata['note']}")

torch.compile disabled: unsupported on Python 3.14


/Users/juan/Juan/Project/Git/HackNusa/backend/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: laya_v3 (from .env)
fine-tuned on an unrecorded number of pages, best epoch 1
  no held-out test split: metadata['test_metrics'] is empty
  validation split only (selected this checkpoint, so optimistic): AUC 0.9573625, accuracy 0.88375, n=800
  status: uncalibrated_recovery; calibrated=False, threshold_selected=False
  note: Emergency inference artifact only. Calibration, test and reload parity are pending.


## The question, and what the model is shown

For a fine-tuned checkpoint the question is not a free choice: the head was trained against
one specific wording, so it is read back from `metadata.json`. For the base checkpoint the
wording is the measurement, and changing it changes the numbers.

Both models get HTML cleaned the way MarkupLM `v1` cleans it — scripts, styles and comments
removed, 300k characters at most — so no model is handed a different page.

Cleaned markup beats visible text alone, measured on the handwritten examples with the base
checkpoint: 11/14 against 10/14, and a wider gap between the classes (0.824 vs 0.295 with
markup, 0.952 vs 0.456 with text). Tags cost tokens but carry the `<form>` and
`<input type=password>` elements that give a credential trap away.


In [2]:
# A fine-tuned head answers the question it was trained on; take it from metadata when present.
trained_question = metadata.get("question") or {}
INSTRUCTIONS = trained_question.get("ins") or (
    "Is this web page a phishing page that impersonates a real brand to steal user credentials?")
QUESTION = {"phishing": {"type": "noul", "instructions": INSTRUCTIONS}}
print("question:", INSTRUCTIONS)
print("source  :", "metadata.json (matches training)" if trained_question else "notebook default")

# Preprocessing follows the checkpoint's own record when it has one.
_prep = metadata.get("preprocessing") or {}
PARSE_CHAR_BUDGET = int(_prep.get("parse_char_budget", 300_000))
STRIP_TAGS = list(_prep.get("strip_tags", ["script", "style", "noscript", "template"]))
THRESHOLD = float((metadata.get("inference") or {}).get("operating_threshold", 0.5))
print(f"cleaning: strip {STRIP_TAGS}, budget {PARSE_CHAR_BUDGET:,} chars, threshold {THRESHOLD}")


def clean_html(html):
    soup = BeautifulSoup(html[:PARSE_CHAR_BUDGET], "html.parser")
    for element in soup.find_all(STRIP_TAGS):
        element.decompose()
    for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
        comment.extract()
    return str(soup)


def predict_html(html):
    """Mirrors predict_html in smoke_test.ipynb so the two notebooks stay comparable."""
    if not isinstance(html, str):
        raise TypeError("html must be a string")
    cleaned = clean_html(html)
    if not cleaned.strip():
        raise ValueError("No usable markup")
    result = agent.predict(cleaned, QUESTION)
    score = float(result["answers"]["phishing"]["noul"])
    return {
        "prediction": "phishing" if score >= THRESHOLD else "benign",
        "phishing_score": score,
        "benign_score": 1.0 - score,
        "threshold": THRESHOLD,
        "html_chars": len(html),
        "input_tokens": int(result.get("usage", {}).get("input_tokens", 0)),
    }


_sample = Path("examples/phishing/klik_bca.html")
demo = predict_html(_sample.read_text(encoding="utf-8") if _sample.exists()
                    else "<html><body><form><input type=password></form></body></html>")
print(json.dumps(demo, indent=2))

question: Is this web page a phishing page that impersonates a real brand to steal user credentials?
source  : metadata.json (matches training)
cleaning: strip ['script', 'style', 'noscript', 'template'], budget 300,000 chars, threshold 0.5


{
  "prediction": "phishing",
  "phishing_score": 0.9854,
  "benign_score": 0.014599999999999946,
  "threshold": 0.5,
  "html_chars": 5784,
  "input_tokens": 837
}


## Handwritten diagnostics

The same hand-labelled files `smoke_test.ipynb` uses, so the two notebooks can be read side
by side. MarkupLM `v1` scored 12 of 14 back when the folder held 14 files; it has grown
since, so treat that as history rather than as a line to compare against today's count.
A few dozen examples decide nothing on their own — they only show whether the model is
behaving sanely before the larger corpus runs.


In [3]:
EXAMPLES_DIR = Path("examples")
if not EXAMPLES_DIR.is_absolute():
    candidates = [Path.cwd() / EXAMPLES_DIR,
                  Path.cwd() / "ai" / "markuplm" / EXAMPLES_DIR,
                  Path.cwd() / "markuplm" / EXAMPLES_DIR]
    EXAMPLES_DIR = next((p for p in candidates
                         if (p / "benign").is_dir() and (p / "phishing").is_dir()), candidates[0])
EXAMPLES_DIR = EXAMPLES_DIR.resolve()

# The folder sets the expected label; the model never gets a say in its own ground truth.
example_files = [(path, label)
                 for folder, label in (("benign", "benign"), ("phishing", "phishing"))
                 for path in sorted((EXAMPLES_DIR / folder).glob("*.html"))]
if not example_files:
    raise FileNotFoundError(f"No diagnostic HTML under {EXAMPLES_DIR}")

passed = 0
for path, expected in example_files:
    name = path.relative_to(EXAMPLES_DIR).as_posix()
    try:
        result = predict_html(path.read_text(encoding="utf-8", errors="replace"))
        ok = result["prediction"] == expected
        passed += ok
        print(("PASS" if ok else "FAIL"), f"{name:42}",
              f"score={result['phishing_score']:.3f} tokens={result['input_tokens']}")
    except (OSError, ValueError, RuntimeError) as exc:
        print("ERROR", name, exc)
print(f"\n{passed}/{len(example_files)} handwritten examples passed")

Token indices sequence length is longer than the specified maximum sequence length for this model (40156 > 8192). Running this sequence through the model will result in indexing errors


FAIL benign/animepahe.html                      score=0.564 tokens=1024


PASS benign/detikcom.html                       score=0.318 tokens=1024


FAIL benign/elamigosedition.html                score=0.666 tokens=1024


PASS benign/fitgirl-donation.html               score=0.320 tokens=1024


...

PASS phishing/netflix_website.html              score=0.782 tokens=695


PASS phishing/parcel-track.html                 score=0.977 tokens=636


PASS phishing/siteboard-workspace.html          score=0.884 tokens=1024


PASS phishing/tokopedia_checkout.html           score=0.982 tokens=1024


PASS phishing/workspace-document.html           score=0.945 tokens=564

30/33 handwritten examples passed


## Cross-dataset run

The 300 Mendeley pages built by `scripts/download_mendeley_html.py`. Laya never saw this
corpus, and it never saw PhreshPhish either, so unlike MarkupLM `v1` there is no home
advantage to lose — whatever it scores here is simply what a general model does cold.

The trivial baseline is recomputed on the same pages. Anything at or below it means the
answer carries no information beyond "shorter pages look suspicious".


In [4]:
import time

CORPUS_DIR = EXAMPLES_DIR / "mendeley"
if not (CORPUS_DIR / "benign").is_dir() or not (CORPUS_DIR / "phishing").is_dir():
    raise FileNotFoundError(
        f"No corpus at {CORPUS_DIR}. Build one first:\n"
        "    python scripts/download_mendeley_html.py --per-class 150")

corpus_files = [(path, label)
                for folder, label in (("benign", "benign"), ("phishing", "phishing"))
                for path in sorted((CORPUS_DIR / folder).glob("*.html"))]
print(f"{CORPUS_DIR.name}: {len(corpus_files)} pages (roughly {len(corpus_files) * 1.5 / 60:.0f} min)")


def roc_auc(positive_scores, negative_scores):
    """Rank-based AUC, tie-corrected. 0.5 is a coin flip, 1.0 is perfect."""
    if not positive_scores or not negative_scores:
        return float("nan")
    merged = sorted([(s, 1) for s in positive_scores] + [(s, 0) for s in negative_scores])
    ranks, index = {}, 0
    while index < len(merged):
        end = index
        while end + 1 < len(merged) and merged[end + 1][0] == merged[index][0]:
            end += 1
        average = (index + end) / 2 + 1
        for position in range(index, end + 1):
            ranks[position] = average
        index = end + 1
    rank_sum = sum(ranks[i] for i, (_, label) in enumerate(merged) if label == 1)
    n_pos, n_neg = len(positive_scores), len(negative_scores)
    return (rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


model_scores = {"phishing": [], "benign": []}
length_scores = {"phishing": [], "benign": []}
counts = {"tp": 0, "fp": 0, "tn": 0, "fn": 0}
failures = []
started = time.time()

for position, (path, expected) in enumerate(corpus_files, start=1):
    try:
        result = predict_html(path.read_text(encoding="utf-8", errors="replace"))
    except (OSError, ValueError, RuntimeError) as exc:
        failures.append(path.name)
        continue
    model_scores[expected].append(result["phishing_score"])
    # Shorter means more suspicious, negated so higher still means "more phishing".
    length_scores[expected].append(-result["html_chars"])
    if result["phishing_score"] >= THRESHOLD:
        counts["tp" if expected == "phishing" else "fp"] += 1
    else:
        counts["fn" if expected == "phishing" else "tn"] += 1
    if position % 50 == 0:
        print(f"  {position}/{len(corpus_files)}  {time.time() - started:.0f}s", flush=True)

scored = sum(counts.values())
if scored == 0:
    raise RuntimeError("Every page failed; nothing to report.")

accuracy = (counts["tp"] + counts["tn"]) / scored
precision = counts["tp"] / (counts["tp"] + counts["fp"]) if counts["tp"] + counts["fp"] else 0.0
recall = counts["tp"] / (counts["tp"] + counts["fn"]) if counts["tp"] + counts["fn"] else 0.0
laya_auc = roc_auc(model_scores["phishing"], model_scores["benign"])
length_auc = roc_auc(length_scores["phishing"], length_scores["benign"])

print(f"\nscored {scored} pages in {time.time() - started:.0f}s, {len(failures)} unusable")
print(f"accuracy  {accuracy:.3f}   precision {precision:.3f}   recall {recall:.3f}")
print("\nconfusion          predicted phishing   predicted benign")
print(f"  truly phishing   {counts['tp']:>18}   {counts['fn']:>16}")
print(f"  truly benign     {counts['fp']:>18}   {counts['tn']:>16}")
label = "Laya, fine-tuned" if FINE_TUNED else "Laya, zero-shot"
print("\n                                AUC")
print(f"  {label:26}  {laya_auc:.3f}")
print(f"  {'page length alone':26}  {length_auc:.3f}   <- the floor worth clearing")
print(f"  {'MarkupLM v1, fine-tuned':26}  0.814   <- same pages, 756-page sample")
if FINE_TUNED:
    print(f"  {'Laya before fine-tuning':26}  0.724   <- same pages, same question")

if laya_auc <= length_auc:
    print("\nThis checkpoint adds nothing over counting characters on these pages.")
elif FINE_TUNED:
    gain = laya_auc - 0.724
    verdict = ("ahead of MarkupLM v1" if laya_auc > 0.814 else
               f"behind MarkupLM v1 by {0.814 - laya_auc:.3f}")
    print(f"\nFine-tuning moved it {gain:+.3f} AUC against the same pages and the same question, "
          f"and it lands {verdict}. This corpus is the fair comparison: both models are scored "
          "on identical pages, and neither trained on any of them.")
elif laya_auc < 0.814:
    print(f"\nBelow the fine-tuned model by {0.814 - laya_auc:.3f} AUC, which is the price of "
          "never having seen a phishing page. It still beats the trivial baseline.")
else:
    print("\nA model never trained on phishing matched or beat the fine-tuned one, which "
          "says more about how little the fine-tuning data taught than about Laya.")

if failures:
    print(f"\nunusable: {failures[:5]}")


mendeley: 300 pages (roughly 8 min)


  50/300  24s


  100/300  47s


  150/300  72s


  200/300  92s


  250/300  112s


  300/300  134s



scored 300 pages in 134s, 0 unusable
accuracy  0.667   precision 0.784   recall 0.460

confusion          predicted phishing   predicted benign
  truly phishing                   69                 81
  truly benign                     19                131

                                AUC
  Laya, fine-tuned            0.798
  page length alone           0.753   <- the floor worth clearing
  MarkupLM v1, fine-tuned     0.814   <- same pages, 756-page sample
  Laya before fine-tuning     0.724   <- same pages, same question

Fine-tuning moved it +0.074 AUC against the same pages and the same question, and it lands behind MarkupLM v1 by 0.016. This corpus is the fair comparison: both models are scored on identical pages, and neither trained on any of them.


## Where to cut the score

AUC says how well the scores *rank* pages; it says nothing about where to draw the line.
This checkpoint ships `"threshold_selected": false`, so 0.5 is an assumption, and Phisang
does not even use 0.5 — `app/risk.py` labels a page phishing from **0.6** (`MALICIOUS_FROM`),
and the extension blocks from there too. So the row that matters operationally is 0.6, not
the 0.5 the cell above reported.

The sweep reuses the scores already computed, so it costs nothing. Picking a threshold on
this corpus and then reporting that corpus's numbers would be cheating; treat the best row
as "what a tuned cut could look like", to be confirmed on a split this model has not met.


In [5]:
PRODUCTION_THRESHOLD = 0.6   # app/risk.py MALICIOUS_FROM, used by the API and the extension

def at_threshold(cut):
    tp = sum(1 for s in model_scores["phishing"] if s >= cut)
    fn = len(model_scores["phishing"]) - tp
    fp = sum(1 for s in model_scores["benign"] if s >= cut)
    tn = len(model_scores["benign"]) - fp
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"cut": cut, "tp": tp, "fn": fn, "fp": fp, "tn": tn,
            "accuracy": (tp + tn) / max(1, tp + tn + fp + fn),
            "precision": precision, "recall": recall, "f1": f1,
            # Balanced accuracy, so a corpus with uneven classes cannot flatter a cut.
            "balanced": (recall + (tn / max(1, tn + fp))) / 2}

sweep = [at_threshold(round(c / 100, 2)) for c in range(5, 100, 5)]
best_f1 = max(sweep, key=lambda r: r["f1"])
best_balanced = max(sweep, key=lambda r: r["balanced"])

print(f"{'cut':>6}  {'acc':>6} {'prec':>6} {'recall':>6} {'F1':>6} {'bal':>6}   missed phishing")
for row in sweep:
    marks = []
    if abs(row["cut"] - THRESHOLD) < 1e-9: marks.append("metadata")
    if abs(row["cut"] - PRODUCTION_THRESHOLD) < 1e-9: marks.append("PHISANG USES THIS")
    if row is best_f1: marks.append("best F1")
    if row is best_balanced: marks.append("best balanced")
    print(f"{row['cut']:>6.2f}  {row['accuracy']:>6.3f} {row['precision']:>6.3f} "
          f"{row['recall']:>6.3f} {row['f1']:>6.3f} {row['balanced']:>6.3f}   "
          f"{row['fn']:>3}/{row['tp'] + row['fn']}   {' | '.join(marks)}")

shipped = at_threshold(PRODUCTION_THRESHOLD)
print(f"\nAt the cut Phisang actually ships ({PRODUCTION_THRESHOLD}): it catches "
      f"{shipped['recall']:.0%} of the phishing pages here and misses {shipped['fn']} of "
      f"{shipped['tp'] + shipped['fn']}, while calling {shipped['fp']} of "
      f"{shipped['fp'] + shipped['tn']} benign pages phishing.")
print(f"Best F1 on this corpus is at {best_f1['cut']:.2f} "
      f"(recall {best_f1['recall']:.3f}, precision {best_f1['precision']:.3f}) — a candidate "
      "to verify elsewhere, not a result to quote.")


   cut     acc   prec recall     F1    bal   missed phishing
  0.05   0.633  0.580  0.967  0.725  0.633     5/150   
  0.10   0.713  0.650  0.927  0.764  0.713    11/150   best F1
  0.15   0.697  0.669  0.780  0.720  0.697    33/150   
  0.20   0.707  0.694  0.740  0.716  0.707    39/150   
  0.25   0.717  0.737  0.673  0.704  0.717    49/150   
  0.30   0.720  0.758  0.647  0.698  0.720    53/150   best balanced
  0.35   0.717  0.798  0.580  0.672  0.717    63/150   
  0.40   0.703  0.796  0.547  0.648  0.703    68/150   
  0.45   0.683  0.789  0.500  0.612  0.683    75/150   
  0.50   0.667  0.784  0.460  0.580  0.667    81/150   metadata
  0.55   0.650  0.778  0.420  0.545  0.650    87/150   
  0.60   0.640  0.776  0.393  0.522  0.640    91/150   PHISANG USES THIS
  0.65   0.640  0.784  0.387  0.518  0.640    92/150   
  0.70   0.627  0.806  0.333  0.472  0.627   100/150   
  0.75   0.620  0.846  0.293  0.436  0.620   106/150   
  0.80   0.573  0.789  0.200  0.319  0.573   120/150  